# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KijoSal-dev/flyrank-ml-internship-wk1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [18]:
# connecting HuggingFace and DuckDB
import getpass
import duckdb

# Enter your Hugging Face READ token when prompted.
# The token will not be displayed or saved in this notebook.
HF_TOKEN = getpass.getpass("Enter your Hugging Face READ token (hf_...): ")

if not HF_TOKEN.startswith("hf_"):
    raise ValueError("That does not look like a Hugging Face token.")

# Connect DuckDB
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

# Hugging Face warehouse location
REL = "hf://datasets/FlyRank/internship-warehouse"

# Warehouse tables
TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Hugging Face token accepted.")
print("DuckDB connected.")
print("FlyRank warehouse paths configured.")


Hugging Face token accepted.
DuckDB connected.
FlyRank warehouse paths configured.


In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_engaged_sessions",
    "sessions_organic",
]

features_march = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_engaged_sessions,
        sessions_organic,
        ga4_data_available
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

print(f"Rows: {len(features_march):,}")
print(f"Features: {feature_cols}")

features_march[feature_cols].head()


Rows: 9,841,378
Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_engaged_sessions', 'sessions_organic']


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,sessions_organic
0,20,0,3.350000,<NA>,<NA>
1,1,0,0.000000,<NA>,<NA>
2,125,1,4.928000,<NA>,<NA>
3,7,0,4.000000,<NA>,<NA>
4,11,0,2.272727,<NA>,<NA>


In [20]:
# ML-05 Section 1: inspect and handle missing values

missing_before = features_march[feature_cols].isna().sum()

print("Missing values before handling:")
print(missing_before)

print("\nGA4 availability:")
print(features_march["ga4_data_available"].value_counts(dropna=False))


Missing values before handling:
gsc_impressions               0
gsc_clicks                    0
gsc_avg_position        6230317
ga4_engaged_sessions    3018741
sessions_organic        3018741
dtype: int64

GA4 availability:
ga4_data_available
False    6408671
<NA>     3018741
True      413966
Name: count, dtype: Int64


In [21]:
# ML-05 Section 1: create the cleaned feature vector

feature_frame = features_march[[
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_engaged_sessions",
    "sessions_organic",
    "ga4_data_available"
]].copy()

# Missingness flags preserve the fact that information was unavailable.
feature_frame["has_gsc_position"] = (
    feature_frame["gsc_avg_position"].notna()
).astype(int)

# True = GA4 is available.
# False or <NA> = GA4 is not confirmed available.
feature_frame["has_ga4_data"] = (
    feature_frame["ga4_data_available"]
    .fillna(False)
    .astype(bool)
    .astype(int)
)

# Keep the original missing numeric values.
# We do NOT replace missing values with zero because missingness is meaningful.
print("Feature frame shape:", feature_frame.shape)

print("\nFeature columns:")
print(feature_cols)

print("\nMissing values:")
print(feature_frame[feature_cols].isna().sum())

print("\nMissingness indicators:")
print(
    feature_frame[["has_gsc_position", "has_ga4_data"]]
    .sum()
)

feature_frame[feature_cols].head()


Feature frame shape: (9841378, 10)

Feature columns:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_engaged_sessions', 'sessions_organic']

Missing values:
gsc_impressions               0
gsc_clicks                    0
gsc_avg_position        6230317
ga4_engaged_sessions    3018741
sessions_organic        3018741
dtype: int64

Missingness indicators:
has_gsc_position    3611061
has_ga4_data         413966
dtype: int64


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,sessions_organic
0,20,0,3.350000,<NA>,<NA>
1,1,0,0.000000,<NA>,<NA>
2,125,1,4.928000,<NA>,<NA>
3,7,0,4.000000,<NA>,<NA>
4,11,0,2.272727,<NA>,<NA>


## 1. Build the feature vector

I built a March 2026 feature frame from the daily performance table. The frame contains 9,841,378 rows and five planned performance features: `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_engaged_sessions`, and `sessions_organic`.

I kept `client_hash_id` and `content_hash_id` in the working frame for identifying and joining observations, but they are not model features because they are pseudonymous IDs.

I also created two missingness indicators: `has_gsc_position` and `has_ga4_data`. These indicate whether the corresponding information was available.

The five numeric features were not blindly filled with zero. `gsc_avg_position` is missing for 6,230,317 rows, while `ga4_engaged_sessions` and `sessions_organic` are each missing for 3,018,741 rows. I kept these missing values because missing data can mean that the underlying measurement was unavailable rather than that the observed value was zero.

The resulting feature frame has 10 columns: the two context IDs, five planned features, `ga4_data_available`, and the two missingness indicators.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-05 Section 2: verify feature missingness

feature_notes_check = feature_frame[feature_cols].isna().agg(["sum", "count"]).T

feature_notes_check["missing_pct"] = (
    feature_notes_check["sum"] / feature_notes_check["count"] * 100
).round(2)

feature_notes_check


,sum,count,missing_pct
gsc_impressions,0,9841378,0.00
gsc_clicks,0,9841378,0.00
gsc_avg_position,6230317,9841378,63.31
ga4_engaged_sessions,3018741,9841378,30.67
sessions_organic,3018741,9841378,30.67


## 2. Feature notes (meaning, missing, categorical, available-when?)

I used five numeric features in the initial feature vector:

- `gsc_impressions` — Google Search Console impressions for the content item on the report date. It has 0.00% missing values in the March 2026 slice.
- `gsc_clicks` — Google Search Console clicks for the content item on the report date. It has 0.00% missing values in the March 2026 slice.
- `gsc_avg_position` — measured average Google Search position. It is missing in 63.31% of March rows, so I retain its missingness rather than treating missing as a real ranking value.
- `ga4_engaged_sessions` — GA4 engaged sessions. It is missing in 30.67% of March rows because GA4 is not available for most rows in this slice.
- `sessions_organic` — organic sessions from GA4. It is also missing in 30.67% of March rows for the same availability reason.

The feature vector contains no categorical features at this stage. The identifiers `client_hash_id` and `content_hash_id` are retained only for grouping and traceability and are not model features.

The missingness indicators `has_gsc_position` and `has_ga4_data` record whether the corresponding measurement is available. I use these indicators because missingness is informative in this warehouse: a missing GA4 value does not mean zero engagement.

All selected measurements are intended to represent information available at the observation date. I will verify the timing and test for future or label-derived information in the leakage hunt in Section 3.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: leakage hunt

# Features currently used in the feature vector
selected_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_engaged_sessions",
    "sessions_organic",
]

# Columns that are known to be unsafe because they describe the outcome
# or are derived from the outcome/trend.
known_label_related = [
    "trend_pct",
    "trend_direction",
    "is_declining_label",
]

# Columns that should not be model features because they are identifiers,
# availability/context fields, or other metadata.
context_or_excluded = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available",
]

# Check whether any selected feature is directly label-related.
label_leakage = sorted(set(selected_features) & set(known_label_related))

# Check whether any selected feature is actually an identifier/context field.
context_leakage = sorted(set(selected_features) & set(context_or_excluded))

print("Selected features:")
print(selected_features)

print("\nDirect label-derived features found:")
print(label_leakage)

print("\nContext/identifier fields accidentally selected:")
print(context_leakage)

# Check the observation window used to build the feature vector.
window_check = con.sql(f"""
    SELECT
        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date,
        COUNT(*) AS rows_checked
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

print("\nFeature observation window:")
display(window_check)

# Final simple verdict
if not label_leakage and not context_leakage:
    print("\nLeakage check result: no direct label-derived or identifier/context fields are in the selected feature list.")
else:
    print("\nLeakage check result: review the fields listed above before modeling.")



Selected features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_engaged_sessions', 'sessions_organic']

Direct label-derived features found:
[]

Context/identifier fields accidentally selected:
[]

Feature observation window:


,first_report_date,last_report_date,rows_checked
0,2026-03-01,2026-03-31,9841378



Leakage check result: no direct label-derived or identifier/context fields are in the selected feature list.


## 3. The leakage hunt

I checked the five selected features for direct label leakage and accidental use of identifiers or context fields.

The selected features are `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_engaged_sessions`, and `sessions_organic`.

The leakage check found no direct label-derived features. In particular, `trend_pct`, `trend_direction`, and `is_declining_label` were not included in the feature vector. These fields are excluded because they describe or are derived from the outcome and would not be safe model inputs.

The check also found no identifiers or context fields accidentally included as model features. `client_hash_id`, `content_hash_id`, and other context or availability fields are kept outside the feature vector.

The feature observation window checked was March 1–31, 2026, with 9,841,378 rows observed. This confirms the dates represented by this feature table.

The result is a clean direct-leakage check for the selected features. However, this does not prove that the features are safe for every possible prediction target: the prediction timing and outcome window must also be aligned when the final label is created.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: fields excluded from the feature vector

excluded_fields = {
    "trend_pct": "Excluded because it is used to derive the trend outcome and would leak label information.",
    "trend_direction": "Excluded because it is derived from trend_pct and is directly related to the outcome.",
    "is_declining_label": "Excluded because it is the prediction label, not a feature.",
    "client_hash_id": "Excluded because it is a pseudonymous identifier used for grouping, joining, and splitting.",
    "content_hash_id": "Excluded because it is a pseudonymous identifier used for grouping and joining, not a predictive measurement.",
    "report_date": "Excluded from the model features because it defines the observation time and is used for temporal alignment.",
    "client_has_gsc": "Excluded because it describes data availability/context rather than content performance.",
    "client_has_ga4": "Excluded because it describes data availability/context rather than content performance.",
    "gsc_data_available": "Excluded because it is an availability/context field; it is not itself a performance measurement.",
    "ga4_data_available": "Excluded as a raw model feature because it describes whether GA4 data exists. A separate availability indicator is retained to handle missingness explicitly."
}

print("Excluded fields and reasons:\n")

for field, reason in excluded_fields.items():
    print(f"- {field}: {reason}")

# Verify that none of the excluded fields are in the selected feature vector.
overlap = sorted(set(excluded_fields) & set(selected_features))

print("\nExcluded fields accidentally included in selected features:")
print(overlap)

print("\nNumber of excluded fields:", len(excluded_fields))
print("Number accidentally included:", len(overlap))



Excluded fields and reasons:

- trend_pct: Excluded because it is used to derive the trend outcome and would leak label information.
- trend_direction: Excluded because it is derived from trend_pct and is directly related to the outcome.
- is_declining_label: Excluded because it is the prediction label, not a feature.
- client_hash_id: Excluded because it is a pseudonymous identifier used for grouping, joining, and splitting.
- content_hash_id: Excluded because it is a pseudonymous identifier used for grouping and joining, not a predictive measurement.
- report_date: Excluded from the model features because it defines the observation time and is used for temporal alignment.
- client_has_gsc: Excluded because it describes data availability/context rather than content performance.
- client_has_ga4: Excluded because it describes data availability/context rather than content performance.
- gsc_data_available: Excluded because it is an availability/context field; it is not itself a performa

## 4. What I excluded and why

I excluded 10 fields from the model feature vector because they are labels, label-derived information, identifiers, timing/context fields, or data-availability fields.

`trend_pct`, `trend_direction`, and `is_declining_label` were excluded because they describe or derive the outcome and could leak label information.

`client_hash_id` and `content_hash_id` were excluded because they are pseudonymous identifiers used for grouping, joining, and splitting rather than predictive measurements.

`report_date` was excluded from the model features because it defines the observation time and is needed for temporal alignment.

`client_has_gsc`, `client_has_ga4`, and `gsc_data_available` were excluded because they describe data availability or context rather than content performance.

`ga4_data_available` was also excluded as a raw feature. Instead, the feature vector uses an explicit `has_ga4_data` indicator to represent whether the GA4 measurement is available. This avoids treating unavailable GA4 measurements as zero engagement.

The verification found 0 excluded fields accidentally included in the selected feature vector. This supports the feature-vector definition used in this notebook.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.